/workspace/KG-known0/260410_전체사이클_빈도상위10고정_gpt4_38개rel_IDPPA변형_entity

In [1]:
import pandas as pd
multi_predicate_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/RE-TACRED/Train/MINT/RE_TACRED_full_binary_result.csv')
multi_predicate_df.head(2)

,split,sample_id,text,token,subject,subj_start,subj_end,subj_type,object,obj_start,...,occupation_title,title_held_by_person,title_holder,referent_of,title_holder_of,held_position,marital_partner_of_blogger,profession_of_person,title_or_position_holder,title_assignment
0,train,61b3a65fb9acc5c72e88,"* `` Updates , Dcr -LRB- DCR -RRB- '' < Dcr.Up...","['*', '``', 'Updates', ',', 'Dcr', '-LRB-', 'D...",DCR,6,6,ORGANIZATION,Dcr,4,...,0,0,0,1,0,0,0,0,0,0
1,train,61b3a65fb916e324f16c,-LRB- While I 'm sure that Fidelity also reser...,"['-LRB-', 'While', 'I', ""'m"", 'sure', 'that', ...",Fidelity,14,14,ORGANIZATION,Fidelity,6,...,0,0,0,1,0,0,0,0,0,0


In [2]:
import spacy
import pandas as pd


nlp = spacy.load("en_core_web_sm")

multi_df = multi_predicate_df.copy()


multi_df["predicate"] = [[] for _ in range(len(multi_df))]


def find_entity_span(doc, entity_text):
    """
    doc 안에서 entity_text와 정확히 일치하는 연속 span을 찾아 반환
    없으면 None 반환
    """
    if not isinstance(entity_text, str) or not entity_text.strip():
        return None

    entity_text = entity_text.strip().lower()
    entity_words = entity_text.split()
    doc_words = [token.text.lower() for token in doc]

    n = len(entity_words)
    for i in range(len(doc_words) - n + 1):
        if doc_words[i:i+n] == entity_words:
            return doc[i:i+n]

    return None


def get_connected_verbs_from_span(span):
    """
    엔티티 span의 각 토큰으로부터 head 방향으로 올라가며
    처음 만나는 VERB/AUX를 predicate 후보로 수집
    """
    predicates = set()

    if span is None:
        return []

    for token in span:
        current = token
        visited = set()

        while current is not None and current.i not in visited:
            visited.add(current.i)

            if current.pos_ in {"VERB", "AUX"}:
                predicates.add(current.lemma_.lower())
                break

            if current.head == current:
                break

            current = current.head

    return list(predicates)


def extract_predicates_for_entities(text, subject, object_):
    """
    subject, object 각각과 관련된 술어를 추출
    결과: list[str]
    """
    if not isinstance(text, str) or not text.strip():
        return []

    doc = nlp(text)

    subj_span = find_entity_span(doc, subject)
    obj_span = find_entity_span(doc, object_)

    predicates = set()

    for p in get_connected_verbs_from_span(subj_span):
        predicates.add(p)

    for p in get_connected_verbs_from_span(obj_span):
        predicates.add(p)

    return list(predicates)



multi_df["predicate"] = multi_df.apply(
    lambda row: extract_predicates_for_entities(
        row["text"],
        row["subject"],
        row["object"]
    ),
    axis=1
)

In [3]:
multi_df.head()

,split,sample_id,text,token,subject,subj_start,subj_end,subj_type,object,obj_start,...,title_held_by_person,title_holder,referent_of,title_holder_of,held_position,marital_partner_of_blogger,profession_of_person,title_or_position_holder,title_assignment,predicate
0,train,61b3a65fb9acc5c72e88,"* `` Updates , Dcr -LRB- DCR -RRB- '' < Dcr.Up...","['*', '``', 'Updates', ',', 'Dcr', '-LRB-', 'D...",DCR,6,6,ORGANIZATION,Dcr,4,...,0,0,1,0,0,0,0,0,0,[]
1,train,61b3a65fb916e324f16c,-LRB- While I 'm sure that Fidelity also reser...,"['-LRB-', 'While', 'I', ""'m"", 'sure', 'that', ...",Fidelity,14,14,ORGANIZATION,Fidelity,6,...,0,0,1,0,0,0,0,0,0,[reserve]
2,train,61b3a65fb96a377b22eb,"In my case , it goes something along the lines...","['In', 'my', 'case', ',', 'it', 'goes', 'somet...",Pentax,47,47,ORGANIZATION,Pentax,19,...,0,0,0,0,0,0,0,0,0,[go]
3,dev,e7798a9e3660ae176967,The documents on ghostwriting were uncovered b...,"['The', 'documents', 'on', 'ghostwriting', 'we...",Public Library of Science,28,31,ORGANIZATION,PLoS,20,...,0,1,0,1,0,0,0,0,0,[make]
4,train,61b3a65fb997f51a5aef,Zhang Chunxian has been appointed as the Party...,"['Zhang', 'Chunxian', 'has', 'been', 'appointe...",Communist Party of China,21,24,ORGANIZATION,CPC,26,...,0,1,1,0,1,0,0,1,1,"[appoint, accord]"


In [4]:
multi_df.to_csv('/workspace/KG-known0/ARR2026/Data_paper/RE-TACRED/Train/MINT/RE_TACRED_full_binary_question_result_predicate.csv', index = False)

# predicate 통계

In [5]:
import ast

def count_predicate_items(x):
    
    if isinstance(x, list):
        return len(x)
    
    
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return len(parsed)
        except Exception:
            pass
    
    
    return 0


multi_df["predicate_count"] = multi_df["predicate"].apply(count_predicate_items)


print(multi_df["predicate_count"].describe())


print(multi_df["predicate_count"].value_counts().sort_index())

count    1850.000000
mean        1.308108
std         0.530507
min         0.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         4.000000
Name: predicate_count, dtype: float64
predicate_count
0      59
1    1165
2     624
3       1
4       1
Name: count, dtype: int64
